# သင်ခန်းစာ ၀၅ - Agentic RAG


## Setup

This notebook demonstrates the Agentic RAG (Retrieval-Augmented Generation) pattern using the Microsoft Agent Framework.

**Prerequisites:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — your Azure AI Search service endpoint
- `AZURE_SEARCH_API_KEY` — your Azure AI Search API key
- Azure OpenAI deployment configured via environment variables
- Azure CLI authenticated (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Agentic RAG ဆိုတာဘာလဲ?

ရိုးရာ RAG က document တွေကို ရှာဖွေပြီးနောက် တုံ့ပြန်ချက် generate လုပ်တဲ့ fixed pipeline ကိုလိုက်နာပါတယ်။ **Agentic RAG** ကတော့ agent ကို autonomy ပေးပြီး **ဘယ်အချိန်မှာ** နဲ့ **ဘယ်လို** သတင်းအချက်အလက် ရှာဖွေရမလဲဆိုတာ ပြိုင်တည်ခွင့်ပေးပါတယ်။

Agentic RAG နဲ့ agent က:
- မေးခွန်းက ပြန်ကြားရန်မတိုင်မီ retrieval လိုအပ်မလိုရှိမှုကို **ဆုံးဖြတ်**
- မေးမြန်းရန် ဒေတာရင်းမြစ် သို့မဟုတ် ကိရိယာကို **ရွေးချယ်**
- ရှာဖွေတွေ့ရှိလာမှုများကို **သုံးသပ်**ပြီး ပထမဆုံးကြိုးပမ်းချက် မဖြစ်မနေ အားနည်းရင်နောက်ထပ် ရှာဖွေမှုလုပ်ဆောင်
- စိစစ်ရရှိမှုများ များစွာမှ အသိပညာ တစ်ပြိုင်နက်ဖြင့် **ပေါင်းစည်း**

ဒါက agent ကို စက်တင် retrieve-then-generate pipeline နှင့် နှိုင်းယှဉ်လျှင် ပိုမို အဆင်ပြေပြီး တိကျစေပါတယ်။


## ရှာဖွေရေးကိရိယာတစ်ခုဖန်တီးခြင်း

Agentic RAG တွင် အပြင်မှ ဒေတာအရင်းအမြစ်များကို အေးဂျင့်ကလိုချင်သလို ခေါ်ယူနိုင်သော **ကိရိယာများ** အဖြစ် အလှည့်ကျပတ်သတ်ထားသည်။ ၎င်းကြောင့် အေးဂျင့်သည် ရှာဖွေတာခြင်းကို မဖြစ်မနေလိုက်နာရမည့်အဆင့်မဟုတ်ပဲ ၎င်းလုပ်ဆောင်နိုင်သော အခြားတစ်ခုသော လုပ်ဆောင်ချက်တစ်ခုအဖြစ် ယူဆနို်င်သည်။

အောက်တွင် ခရီးသွားဆိုင်ရာ ဗဟုသုတအုပ်စုတစ်ခုကို သတ်မှတ်ပြီး အေးဂျင့်သည် တည်နေရာသတင်းအချက်အလက်များရှာဖွေရန် ခေါ်ယူနိုင်သော ကိရိယာအဖြစ် ထုတ်ဖော်ပြသထားသည်။


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG Agent ဆောက်ခြင်း

ယခု ကျွန်ုပ်တို့မှာ **ဖြေဆိုမည့်အခါ အမြဲတမ်း သတင်းအချက်အလက်ကို ရှာဖွေယူရန်** တာဝန်ပေးထားသော အေးဂျင့်တစ်ယောက်ကို ဖန်တီးလိုက်ပြီ ဖြစ်သည်။ အဲဒီ Agent သည် သင်၏ကိုယ်ပိုင် လေ့ကျင့်မှုဒေတာအစား သိပ္ပံအခြေခံ ဒေတာအတွက် `search_travel_knowledge` ကိရိယာကို အသုံးပြုပြီး ၎င်း၏ ပြန်ဆိုချက်များကို အခြေပြုသည်။


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## အကြိမ်ကြိမ် ထုတ်ယူခြင်း — Maker-Checker ပုံစံ

Agentic RAG ၏ အဓိကအားသာချက်တစ်ရပ်မှာ **အကြိမ်ကြိမ် ထုတ်ယူခြင်း** ဖြစ်သည်။ အဲ့ဒီ agent က သူ့ရဲ့ ပထမဆုံး ရရှိချက်တွေကို အတည်ပြုဖို့၊ ကောင်းမွန်အောင် ပြုပြင်ဖို့ သို့မဟုတ္ ကောက်နှုတ်ချက်တွေ ထပ်မံတိုးချဲ့ဖို့ ရှာဖွေမှု အကြိမ်အနည်းငယ် ဆောင်ရွက်နိုင်သည် — "maker-checker" workflow နဲ့ ရောနှောပါသည်။

1. **Maker ခြေလှမ်း**: Agent က ပထမဆုံး သတင်းအချက်အလက်တွေကို ရယူပြီး ပြန်လည်တုံ့ပြန်ချက် ပုံဖော်သည်။
2. **Checker ခြေလှမ်း**: Agent က အသေးစိတ်ကို အတည်ပြုရန် သို့မဟုတ် အခွာအဝေးတွေ ဖြည့်ရန် ထပ်မံ ထုတ်ယူမှုများ ဆောင်ရွက်သည်။

အောက်တွင် agent ကို အများအပြား သွားရာဒေသများကို နှိုင်းယှဉ်ဖို့ လိုသော ကိစ္စမေးမြန်းထားပြီး၊ ဒါကြောင့် တချို့ ရှာဖွေရေးများ ထပ်မံ ဆောင်ရွက်ရန် လိုအပ်ခဲ့သည်။


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## အကျဉ်းချုပ်

ဒီသင်ခန်းစာမှာ Microsoft Agent Framework ကို အသုံးပြုပြီး **Agentic RAG** စနစ်တစ်ခု ဖန်တီးနည်းကို သင်ယူခဲ့ပါသည်။

- **Agentic RAG** က ကိုယ်တိုင် ဆုံးဖြတ်ချက်ချ၍ အချက်အလက်တွေ ပြန်ရွေးယူရာမှာ တင်းကျပ်ခြင်းမရှိဘဲ လှုပ်ရှားမှုပြောင်းလဲစေပါတယ်။
- **ကိရိယာများအား ဒေတာအရင်းအမြစ်အဖြစ်**: အပြင်ရှိ အသိပညာဘဏ်များ (Azure AI Search စသဖြင့်) ကို agent မှ ခေါ်နိုင်သော ကိရိယာများအဖြစ်ထုပ်ပိုးထားသည်။
- **အဆင့်ဆင့် ပြန်ရွေးယူခြင်း**: maker-checker အမျိုးအစားပုံစံက agent ကို ရှာဖွေရေး၊ စစ်ဆေးခြင်းနှင့် တိုးတက်မွမ်းမံခြင်း အဆင့်များကို ချိန်ညှိစေပြီး နောက်ဆုံးဖြေချက်ထုတ်နိုင်စေသည်။

ထုတ်လုပ်မှုတွင် မျှော်လင့်ထားသလို ကြီးမားသောခရီးသွားစာရွက်စာတမ်းများကို ခုံတွင်သွင်းထားသော `TRAVEL_KNOWLEDGE_BASE` အစား အမှန်တစ်ကယ် Azure AI Search index ကို အသုံးပြုပါလိမ့်မယ်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
